# Customer Performance Model Training

import libraries

In [1]:
import os
import json
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    r2_score, accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)

Data pre-processing

In [2]:
#import dataset
df = pd.read_csv("data/customer_performance.csv", sep="\t")
print("Original data shape:", df.shape)

# Fill missing income values with the middle (median) income
df["Income"] = df["Income"].fillna(df["Income"].median())

# Remove the requested unusual birth years and high-income outliers
df = df[(df["Year_Birth"] > 1920) & (df["Income"] < 200000)].copy()

# Make age and number of children from the existing columns.
df["Age"] = 2014 - df["Year_Birth"]
df["Children"] = df["Kidhome"] + df["Teenhome"]

# Add together every column whose name starts with Mnt to make the spending target.
mnt_columns = [column for column in df.columns if column.startswith("Mnt")]
df["TotalSpend"] = df[mnt_columns].sum(axis=1)

# Remove identifier and source columns that are not model inputs.
df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue", "Year_Birth", "Dt_Customer"])
print("Rows after cleaning:", df.shape[0])
print("Missing values after cleaning:", df.isna().sum().sum())


Original data shape: (2240, 29)
Rows after cleaning: 2236
Missing values after cleaning: 0


In [3]:
features = [
    "Income", "Age", "Children", "Recency", "NumWebPurchases",
    "NumCatalogPurchases", "NumStorePurchases", "NumWebVisitsMonth",
    "NumDealsPurchases"
]
X = df[features] #both model will use same featues to avoid data leakage

# Split spending data and keep 20% for evaluation
y_reg = df["TotalSpend"]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

# Stratify the response split so the less common accepted class stays represented
y_clf = df["Response"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)


Linear Regression

In [4]:
reg = LinearRegression()
reg.fit(Xr_train, yr_train)
spend_predictions = reg.predict(Xr_test) #predict spending of those custokmers that model havnot seen before
r2 = r2_score(yr_test, spend_predictions) #competion of real or predicted spending
print("R2:", r2)



R2: 0.8072739488501225


Classification

In [5]:
# Scaling is included in the pipeline so it is applied consistently at prediction time.
clf = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000))
])
clf.fit(Xc_train, yc_train)
response_predictions = clf.predict(Xc_test) #tells 0 or 1
response_probabilities = clf.predict_proba(Xc_test)[:, 1] #give class probabilities for the positive class

# Calculate several scores because the response classes are imbalanced.
accuracy = accuracy_score(yc_test, response_predictions)
precision = precision_score(yc_test, response_predictions, zero_division=0)
recall = recall_score(yc_test, response_predictions, zero_division=0)
f1 = f1_score(yc_test, response_predictions, zero_division=0)
roc_auc = roc_auc_score(yc_test, response_probabilities)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC-AUC:", roc_auc)



Accuracy: 0.7388392857142857
Precision: 0.3333333333333333
Recall: 0.746268656716418
F1: 0.4608294930875576
ROC-AUC: 0.8123555451091001


In [6]:
# Make the models folder if it does not already exist.
os.makedirs("models", exist_ok=True)

# Save both trained models so main.py can load them later.
joblib.dump(reg, "models/spending_model.joblib")
joblib.dump(clf, "models/response_model.joblib")

# Keep coefficients with their feature names to make the chart readable.
linear_coefficients = dict(zip(features, reg.coef_.tolist()))
logistic_coefficients = dict(zip(features, clf.named_steps["model"].coef_[0].tolist()))

# Save 100 test customers for the actual-versus-predicted chart.
actual_vs_predicted = [
    {"actual": float(actual), "predicted": float(predicted)}
    for actual, predicted in zip(yr_test.iloc[:100], spend_predictions[:100])
]

# Count how many cleaned customers are in each response class.
response_distribution = {
    str(label): int(count)
    for label, count in df["Response"].value_counts().sort_index().items()
}

# Collect scores and chart data in one JSON-compatible dictionary.
metrics = {
    "r2": float(r2),
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "roc_auc": float(roc_auc),
    "linear_coefficients": linear_coefficients,
    "logistic_coefficients": logistic_coefficients,
    "actual_vs_predicted": actual_vs_predicted,
    "confusion_matrix": confusion_matrix(yc_test, response_predictions, labels=[0, 1]).tolist(),
    "response_distribution": response_distribution,
    "features": features
}

# Write metrics.json so the FastAPI endpoint can return it to the charts.
with open("models/metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics, file, indent=2)

print("Models and metrics saved in the models folder.")


Models and metrics saved in the models folder.
